# 🎓 University Regulation RAG Assistant — Colab Pipeline

This notebook does **Phase 2** of the project:

```
PDF FILES → Extract Text → Clean → Chunk → Embeddings → FAISS Vector DB → Save
```

Every chunk keeps metadata (`source`, `file`, `page`, `category`) so the backend can later show **which document and page** an answer came from, exactly as required.

**How to use this notebook:**
1. Run cells top to bottom.
2. When Cell 3 asks you to upload files, upload your 4–5 university PDFs (Academic Regulations, Examination Rules, Attendance Policy, Student Handbook, Curriculum).
3. At the end you'll get `vector_db/index.faiss` and `vector_db/metadata.pkl` — download these, you'll load them in the FastAPI backend later.

## Step 1 — Install libraries

In [ ]:
!pip install -q pdfplumber langchain sentence-transformers faiss-cpu tiktoken

## Step 2 — Imports

In [ ]:
import os
import re
import json
import pickle
import pdfplumber
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from google.colab import files

## Step 3 — Upload your PDFs

Run this cell, then choose the PDF files from your computer (Academic Regulations, Examination Rules, Attendance Policy, Student Handbook, etc.).

In [ ]:
os.makedirs('data/regulations', exist_ok=True)

uploaded = files.upload()

for fname in uploaded.keys():
    dest = os.path.join('data/regulations', fname)
    with open(dest, 'wb') as f:
        f.write(uploaded[fname])

print('Uploaded files:', list(uploaded.keys()))

## Step 4 — Document registry (metadata)

This mirrors your `documents.json` idea from the plan — one entry per PDF, so we know its title/category/source URL.

**Edit this list** to match the files you just uploaded and fill in real official source URLs where you have them.

In [ ]:
# EDIT ME: map each uploaded filename to its title/category/source
DOCUMENT_REGISTRY = [
    {
        "id": "DOC001",
        "title": "Academic Regulations",
        "filename": "academic_regulations.pdf",   # <-- change to your actual filename
        "category": "Academic",
        "source_url": "OFFICIAL_URL"
    },
    {
        "id": "DOC002",
        "title": "Examination Regulations",
        "filename": "examination_rules.pdf",
        "category": "Examination",
        "source_url": "OFFICIAL_URL"
    },
    {
        "id": "DOC003",
        "title": "Attendance Policy",
        "filename": "attendance_policy.pdf",
        "category": "Attendance",
        "source_url": "OFFICIAL_URL"
    },
    {
        "id": "DOC004",
        "title": "Student Handbook",
        "filename": "student_handbook.pdf",
        "category": "Handbook",
        "source_url": "OFFICIAL_URL"
    }
]

os.makedirs('data/metadata', exist_ok=True)
with open('data/metadata/documents.json', 'w') as f:
    json.dump(DOCUMENT_REGISTRY, f, indent=2)

# quick lookup: filename -> registry entry
REGISTRY_BY_FILENAME = {d['filename']: d for d in DOCUMENT_REGISTRY}
print('Saved data/metadata/documents.json')

## Step 5 — Extract text (page by page)

We keep the **page number** for every piece of text, since it needs to show up later in the answer.

In [ ]:
def extract_pages(pdf_path):
    """Returns a list of {page_number, text} for a single PDF."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            pages.append({"page_number": i, "text": text})
    return pages

pdf_folder = 'data/regulations'
all_pages = {}  # filename -> list of {page_number, text}

for fname in os.listdir(pdf_folder):
    if fname.lower().endswith('.pdf'):
        path = os.path.join(pdf_folder, fname)
        print(f'Extracting: {fname}')
        all_pages[fname] = extract_pages(path)

print('\nDone. Pages extracted per file:')
for fname, pages in all_pages.items():
    print(f'  {fname}: {len(pages)} pages')

## Step 6 — Clean text

Basic cleanup: collapse extra whitespace, strip weird characters, remove empty pages.

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)          # collapse whitespace/newlines
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # drop non-ASCII junk (optional)
    return text.strip()

for fname, pages in all_pages.items():
    for p in pages:
        p['text'] = clean_text(p['text'])

# drop pages that ended up empty (e.g. scanned/blank pages)
for fname in list(all_pages.keys()):
    all_pages[fname] = [p for p in all_pages[fname] if len(p['text']) > 0]

print('Cleaning complete.')

## Step 7 — Chunk text (with metadata attached to every chunk)

Each chunk keeps: `text`, `source` (title), `file`, `page`, `category`, `url` — matching the structure from the plan.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_chunks = []  # list of dicts: text/source/file/page/category/url

for fname, pages in all_pages.items():
    reg = REGISTRY_BY_FILENAME.get(fname, {
        "title": fname, "category": "Unknown", "source_url": ""
    })
    for p in pages:
        if not p['text']:
            continue
        pieces = splitter.split_text(p['text'])
        for piece in pieces:
            all_chunks.append({
                "text": piece,
                "source": reg.get("title", fname),
                "file": fname,
                "page": p['page_number'],
                "category": reg.get("category", "Unknown"),
                "url": reg.get("source_url", "")
            })

print(f'Total chunks created: {len(all_chunks)}')
print('\nSample chunk:')
print(json.dumps(all_chunks[0], indent=2) if all_chunks else 'No chunks yet — check your PDFs.')

## Step 8 — Generate embeddings

Using a free, fast, good-quality open model (`all-MiniLM-L6-v2`) — no API key needed.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [c['text'] for c in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

embeddings = embeddings.astype('float32')
print('Embeddings shape:', embeddings.shape)

## Step 9 — Build the FAISS index

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f'FAISS index built with {index.ntotal} vectors of dimension {dimension}.')

## Step 10 — Save the vector database

This produces `vector_db/index.faiss` (the vectors) and `vector_db/metadata.pkl` (the chunk text + source/page info). Both are needed by the FastAPI backend later.

In [ ]:
os.makedirs('vector_db', exist_ok=True)

faiss.write_index(index, 'vector_db/index.faiss')

with open('vector_db/metadata.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)

print('Saved vector_db/index.faiss and vector_db/metadata.pkl')

## Step 11 — Test it: ask a question

This simulates what the backend will do: embed the question, search FAISS, return the top matching chunks with their source + page.

In [ ]:
def search(query, top_k=3):
    q_vec = model.encode([query], convert_to_numpy=True).astype('float32')
    distances, indices = index.search(q_vec, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        chunk = all_chunks[idx]
        results.append({
            "score": float(dist),
            "text": chunk['text'],
            "source": chunk['source'],
            "page": chunk['page'],
            "file": chunk['file']
        })
    return results

test_query = "What is the minimum attendance required?"
results = search(test_query, top_k=3)

for r in results:
    print(f"[{r['source']} | Page {r['page']}] (distance={r['score']:.3f})")
    print(r['text'][:300], '...')
    print('-' * 60)

## Step 12 — Download the vector database

Run this to download the two files so you can put them into `backend/vector_db/` for the FastAPI app (Phase 3).

In [ ]:
files.download('vector_db/index.faiss')
files.download('vector_db/metadata.pkl')
files.download('data/metadata/documents.json')

---
### ✅ What you have now
- `vector_db/index.faiss` — the FAISS vector index
- `vector_db/metadata.pkl` — chunk text + source/page/category/url for every vector, in the same order as the index
- `data/metadata/documents.json` — your document registry

### Next (Phase 3)
Load these two files in FastAPI (`app/rag.py` / `app/retriever.py`), embed the incoming question the same way (`all-MiniLM-L6-v2`), search the FAISS index, pass the top chunks + their page/source to the LLM as context, and return `{answer, source, page, document}` — with a fallback to "Information Not Found" when nothing relevant is retrieved (e.g. thresholding on distance).